# Prepare Toronto ADA Wide Attributes

Join census data from `toronto-ada.csv` with ADA geometries from `toronto-ada.gpkg`
to produce a single GeoPackage and GeoJSON where each ADA feature has one attribute
column per census variable.

Values use `C10_RATE_TOTAL` (rate/percentage, `_pct` suffix) by default, except for
fields that are inherently counts or absolute values, which use `C1_COUNT_TOTAL` (`_count` suffix).

In [10]:
import os

import pandas as pd
import geopandas as gpd
from tqdm import tqdm

## Constants

In [11]:
# Characteristic IDs to include in the wide output.
# These are the active entries from CHARACTERISTIC_CODES in extract_census_ada.ipynb.
# Format: ID,  # <code> -- <full census characteristic name>
ACTIVE_IDS = [
    # Demographics - Population
    1,    # pop_2021 -- Population, 2021
    2,    # pop_2016 -- Population, 2016
    3,    # pop_pct_change -- Population percentage change, 2016 to 2021
    6,    # pop_density -- Population density per square kilometre

    # Age structure
    35,   # age_0_14 -- 0 to 14 years
    36,   # age_15_64 -- 15 to 64 years
    37,   # age_65_over -- 65 years and over
    39,   # age_mean -- Average age of the population
    40,   # age_median -- Median age of the population

    # Household composition
    57,   # pvt_house_avg_size -- Average household size

    # Income - Core measures
    115,  # income_aftertax_median -- Median after-tax income in 2020 among recipients ($)
    122,  # income_ei_recip -- Number of employment insurance benefits recipients
    124,  # income_covid_recip -- Number of COVID-19 emergency and recovery benefits recipients

    # Income - Poverty and distribution
    345,  # lim_at_prev -- Prevalence of low income based on LIM-AT (%)
    346,  # lim_at_0_17 -- LIM-AT prevalence, age 0 to 17 (%)
    347,  # lim_at_0_5 -- LIM-AT prevalence, age 0 to 5 (%)
    348,  # lim_at_18_64 -- LIM-AT prevalence, age 18 to 64 (%)
    349,  # lim_at_65_over -- LIM-AT prevalence, age 65 years and over (%)
    378,  # gini_measures -- Total - Inequality measures (population count used for Gini)
    379,  # gini_total_income -- Gini index on adjusted household total income
    380,  # gini_market_income -- Gini index on adjusted household market income
    381,  # gini_aftertax_income -- Gini index on adjusted household after-tax income
    382,  # gini_p90_p10 -- P90/P10 ratio on adjusted household after-tax income

    # Housing - Tenure and affordability
    1415, # housing_tenure_owner -- Owner
    1416, # housing_tenure_renter -- Renter
    1417, # housing_tenure_provided -- Dwelling provided by local government, First Nation or Indian band
    1418, # housing_condo_total -- Total - Occupied private dwellings by condominium status
    1466, # housing_shelter_under30 -- Spending less than 30% of income on shelter costs
    1467, # housing_shelter_30plus -- Spending 30% or more of income on shelter costs
    1468, # housing_shelter_30_100 -- Spending 30% to less than 100% of income on shelter costs

    # Housing - Core need
    1480, # housing_core_need_yes -- In core housing need (yes)
    1481, # housing_core_need_no -- Not in core housing need (no)
    1483, # housing_owner_mortgage_pct -- % of owner households with a mortgage
    1484, # housing_owner_shelter_30plus_pct -- % of owner households spending 30%+ on shelter costs
    1485, # housing_owner_core_need_pct -- % in core housing need

    # Citizenship
    1523, # citizen_canadian -- Canadian citizens
    1524, # citizen_canadian_under18 -- Canadian citizens aged under 18
    1525, # citizen_canadian_18over -- Canadian citizens aged 18 and over
    1526, # citizen_not_canadian -- Not Canadian citizens

    # Visible minorities
    1684, # visible_minority_yes -- Total visible minority population (yes)
    1697, # visible_minority_no -- Not a visible minority

    # Education - all ages 15+
    1999, # education_none -- No certificate, diploma or degree
    2000, # education_secondary -- High (secondary) school diploma or equivalency certificate
    2001, # education_postsec -- Postsecondary certificate, diploma or degree
    2008, # education_bachelor_higher -- Bachelor's degree or higher

    # Education - ages 25 to 64
    2015, # education_25_64_none -- No certificate, diploma or degree (age 25-64)
    2016, # education_25_64_secondary -- High (secondary) school diploma or equivalency certificate (age 25-64)
    2017, # education_25_64_postsec -- Postsecondary certificate, diploma or degree (age 25-64)
    2024, # education_25_64_bachelor_higher -- Bachelor's degree or higher (age 25-64)

    # Labour - Occupation
    2254, # labour_occupation_5_arts_culture -- Occupation 5 - Arts, culture, recreation and sport

    # Labour - Industry
    2278, # labour_industry_71_arts -- Industry 71 - Arts, entertainment and recreation
]

# Characteristic codes whose values are absolute counts or index values (not rates).
# These use C1_COUNT_TOTAL with a _count suffix; all others use C10_RATE_TOTAL with _pct.
COUNT_ONLY_CODES = {
    'pop_2021',               # Raw population count
    'pop_2016',               # Raw population count
    'age_mean',               # Mean age in years (not a rate)
    'age_median',             # Median age in years (not a rate)
    'pvt_house_avg_size',     # Average household size (not a rate)
    'income_aftertax_median', # Median after-tax income in $ (not a rate)
    'gini_total_income',      # Gini index value
    'gini_market_income',     # Gini index value
    'gini_aftertax_income',   # Gini index value
    'gini_p90_p10',           # P90/P10 ratio value
    'housing_condo_total',    # Raw count of condominium dwellings
}


In [12]:
# File paths
INPUT_CSV  = '../../data/census/ada/toronto-ada.csv'
INPUT_GPKG = '../../data/geo/toronto-ada.gpkg'
OUTPUT_DIR = '../../data/census/ada-wide'

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Load and Pivot Census Data

In [13]:
# Load toronto-ada.csv and filter to active IDs only
df = pd.read_csv(INPUT_CSV, dtype={'GEO_NAME': str, 'CHARACTERISTIC_CODE': str}, encoding='latin')
df['CHARACTERISTIC_ID'] = pd.to_numeric(df['CHARACTERISTIC_ID'], errors='coerce')

df = df[df['CHARACTERISTIC_ID'].isin(ACTIVE_IDS)].copy()
print(f"Rows after filtering: {len(df)}  ({df['GEO_NAME'].nunique()} ADAs, {df['CHARACTERISTIC_ID'].nunique()} variables)")

Rows after filtering: 14229  (279 ADAs, 51 variables)


In [14]:
# Determine output column name and value for each row.
#   - COUNT_ONLY_CODES  → C1_COUNT_TOTAL,  suffix _count
#   - all others        → C10_RATE_TOTAL,  suffix _pct
#     (if C10 is null, fall back to C1_COUNT_TOTAL with _count)

df['C10_RATE_TOTAL'] = pd.to_numeric(df['C10_RATE_TOTAL'], errors='coerce')
df['C1_COUNT_TOTAL'] = pd.to_numeric(df['C1_COUNT_TOTAL'], errors='coerce')

is_count_only = df['CHARACTERISTIC_CODE'].isin(COUNT_ONLY_CODES)
c10_null      = df['C10_RATE_TOTAL'].isna()
use_count     = is_count_only | c10_null

df['col_name'] = df['CHARACTERISTIC_CODE'] + '_pct'
df['value']    = df['C10_RATE_TOTAL']

df.loc[use_count, 'col_name'] = df.loc[use_count, 'CHARACTERISTIC_CODE'] + '_count'
df.loc[use_count, 'value']    = df.loc[use_count, 'C1_COUNT_TOTAL']

# Pivot: one row per ADA, one column per variable
df_wide = df.pivot(index='GEO_NAME', columns='col_name', values='value').reset_index()
df_wide.columns.name = None

print(f"Wide table shape: {df_wide.shape}  ({len(df_wide)} ADAs x {df_wide.shape[1]-1} attributes)")
df_wide.head(2)

Wide table shape: (279, 52)  (279 ADAs x 51 attributes)


,GEO_NAME,age_0_14_pct,age_15_64_pct,age_65_over_pct,age_mean_count,age_median_count,citizen_canadian_18over_pct,citizen_canadian_pct,citizen_canadian_under18_pct,citizen_not_canadian_pct,...,lim_at_18_64_pct,lim_at_65_over_pct,lim_at_prev_pct,pop_2016_count,pop_2021_count,pop_density_pct,pop_pct_change_pct,pvt_house_avg_size_count,visible_minority_no_pct,visible_minority_yes_pct
0,35200001,17.0,71.2,11.8,38.3,37.2,67.8,88.4,20.8,11.5,...,4.2,5.2,5.0,5806.0,5547.0,278.6,-4.5,3.6,5.1,94.9
1,35200002,16.7,70.0,13.3,38.9,38.0,67.3,86.8,19.5,13.2,...,4.0,7.1,5.3,12692.0,12479.0,1964.4,-1.7,4.1,1.8,98.2


## Join with Geometry and Export

In [15]:
# Load ADA geometries — keep only ADAUID + geometry (drops Stats Canada metadata columns
# like ADANAME, PRUID, CSDUID etc. which inflates output file size without adding value)
gdf = gpd.read_file(INPUT_GPKG)
print(f"Geometry CRS: {gdf.crs}")
print(f"Geometry columns (all): {gdf.columns.tolist()}")

gdf = gdf[['ADAUID', 'geometry']]

# Join census wide table on ADAUID == GEO_NAME
gdf_wide = gdf.merge(df_wide, left_on='ADAUID', right_on='GEO_NAME', how='left')
gdf_wide = gdf_wide.drop(columns=['GEO_NAME'])  # drop the duplicate join key

print(f"\nJoined GDF shape: {gdf_wide.shape}")
print(f"Unmatched ADAs: {gdf_wide['pop_2021_count'].isna().sum()}")


Geometry CRS: EPSG:3347
Geometry columns (all): ['ADAUID', 'DGUID', 'LANDAREA', 'PRUID', 'geometry']

Joined GDF shape: (279, 53)
Unmatched ADAs: 0


In [16]:
# Save GeoPackage (original CRS) and GeoJSON (WGS84)
gpkg_path    = os.path.join(OUTPUT_DIR, 'toronto-ada-wide.gpkg')
geojson_path = os.path.join(OUTPUT_DIR, 'toronto-ada-wide.geojson')

print('Saving GeoPackage...')
gdf_wide.to_file(gpkg_path, driver='GPKG')

print('Saving GeoJSON (reprojected to WGS84)...')
# COORDINATE_PRECISION=5 gives ~1m accuracy (vs full double precision default),
# which is more than sufficient for ADA-level census boundaries and significantly
# reduces GeoJSON file size.
gdf_wide.to_crs('EPSG:4326').to_file(geojson_path, driver='GeoJSON', COORDINATE_PRECISION=5)

print(f'\nOutputs written to {OUTPUT_DIR}/')
print(f'  {os.path.basename(gpkg_path)}    ({os.path.getsize(gpkg_path)/1024:.0f} KB)')
print(f'  {os.path.basename(geojson_path)} ({os.path.getsize(geojson_path)/1024:.0f} KB)')


Saving GeoPackage...
Saving GeoJSON (reprojected to WGS84)...

Outputs written to ../../data/census/ada-wide/
  toronto-ada-wide.gpkg    (3412 KB)
  toronto-ada-wide.geojson (5008 KB)


---
## Future: NOC / NAICS Occupation & Industry Data

When NAICS/NOC venue or worker data is available, merge it here into `gdf_wide` before saving,
joining on ADA identifier.

In [17]:
# TODO: merge NOC/NAICS data
# noc_df = pd.read_csv('...')
# gdf_wide = gdf_wide.merge(noc_df, on='ADAUID', how='left')

---
## Future: Activity Data by Venue

When venue activity data (stops, visits, etc.) has been aggregated to the ADA level,
merge it here before saving.

In [18]:
# TODO: merge venue activity data
# activity_df = pd.read_csv('...')
# gdf_wide = gdf_wide.merge(activity_df, on='ADAUID', how='left')